In [1]:
import pandas as pd
import glob
import os

# Importing data
Data folder contains 5 months data for a medium cosmetic online store. Each row represents an event related to products and users. 

Data taken from kaggle. More about dataset you can read [here](https://www.kaggle.com/datasets/mkechinov/ecommerce-events-history-in-cosmetics-shop?select=2019-Dec.csv).

### Combining into one dataframe and investigating details.

I initially downcasted IDs to int32 for memory optimization, but discovered overflow in category_id due to large values. I fixed it by switching to int64 after validating the value range.

In [2]:
path = r'./data'
all_files = glob.glob(os.path.join(path, '*.csv'))

dtypes = {
    'product_id':'int32',
    'category_id':'int64',
    'user_id':'int32',
}

all_data_df = pd.concat(
    (pd.read_csv(f, dtype=dtypes) for f in all_files),
    ignore_index=True,
    )



In [3]:
all_data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20692840 entries, 0 to 20692839
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   event_time     object 
 1   event_type     object 
 2   product_id     int32  
 3   category_id    int64  
 4   category_code  object 
 5   brand          object 
 6   price          float64
 7   user_id        int32  
 8   user_session   object 
dtypes: float64(1), int32(2), int64(1), object(5)
memory usage: 1.2+ GB


In [4]:
all_data_df.sample(5)

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
2669066,2019-12-21 23:00:16 UTC,view,5877612,1487580007675986893,NaN,bpw.style,0.79,584515763,d6277dd3-f08a-4c24-8086-85e500d15eda
11812402,2019-10-28 12:19:03 UTC,cart,5731697,1495705810704007729,NaN,NaN,1.59,476072298,4ff65508-e308-48f3-9882-3db236ebf678
10146726,2019-10-15 06:51:29 UTC,view,5818361,1487580009336930331,NaN,igrobeauty,1.03,560111020,202235e4-0425-4c76-a32c-ff3262fc3873
772904,2019-12-06 20:38:30 UTC,purchase,5911251,1487580007675986893,NaN,dartnails,1.43,583077727,6f2d9fc0-877b-4b4e-9320-d19372e62c92
13159606,2020-02-06 15:24:32 UTC,cart,5649174,1487580005511725929,NaN,NaN,4.27,584163223,4d377819-b343-47c1-bd16-03fdf9ffb456


In [5]:
all_data_df.describe()

,product_id,category_id,price,user_id
count,2.069284e+07,2.069284e+07,2.069284e+07,2.069284e+07
mean,5.484297e+06,1.554230e+18,8.534735e+00,5.215527e+08
std,1.305716e+06,1.691038e+17,1.938142e+01,8.744312e+07
min,3.752000e+03,1.487580e+18,-7.937000e+01,4.654960e+05
25%,5.724650e+06,1.487580e+18,2.060000e+00,4.818306e+08
50%,5.810720e+06,1.487580e+18,4.050000e+00,5.531297e+08
75%,5.857864e+06,1.487580e+18,7.060000e+00,5.788573e+08
max,5.932595e+06,2.242903e+18,3.277800e+02,6.220902e+08


Data contains NaN values and negative prices. Some of data can be categorical like event_type, category_code, or brand.

### Dropping duplicates and counting NaN values.

In [6]:
df_removed_duplicates = all_data_df.drop_duplicates()

In [7]:
df_removed_duplicates.isna().sum()

event_time              0
event_type              0
product_id              0
category_id             0
category_code    19240810
brand             8265149
price                   0
user_id                 0
user_session         4085
dtype: int64

In [8]:
df_removed_duplicates.isna().mean().mul(100)


event_time        0.000000
event_type        0.000000
product_id        0.000000
category_id       0.000000
category_code    98.248894
brand            42.204135
price             0.000000
user_id           0.000000
user_session      0.020859
dtype: float64

As data category_code contains more than 98% empty values, dropping column.

User session is important part of data analysis and ~0.02% is unissential, so removing such rows.

Brand can be envestigated further, so leaving column and replacing NaN values by 'unknown'.

In [9]:
df_cleaned = df_removed_duplicates.drop('category_code', axis=1)

In [10]:
df_cleaned = df_cleaned.dropna(subset=['user_session'])

In [11]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19579657 entries, 0 to 20692839
Data columns (total 8 columns):
 #   Column        Dtype  
---  ------        -----  
 0   event_time    object 
 1   event_type    object 
 2   product_id    int32  
 3   category_id   int64  
 4   brand         object 
 5   price         float64
 6   user_id       int32  
 7   user_session  object 
dtypes: float64(1), int32(2), int64(1), object(4)
memory usage: 1.2+ GB


In [12]:
df_cleaned.isna().sum()


event_time            0
event_type            0
product_id            0
category_id           0
brand           8263669
price                 0
user_id               0
user_session          0
dtype: int64

In [13]:
df_cleaned.isna().mean().mul(100)

event_time       0.000000
event_type       0.000000
product_id       0.000000
category_id      0.000000
brand           42.205382
price            0.000000
user_id          0.000000
user_session     0.000000
dtype: float64

In [14]:
df_cleaned.brand.value_counts()

brand
runail       1433792
irisk         970331
grattol       811021
masura        801532
bpw.style     409838
              ...   
shifei             9
vl-gel             7
dessata            6
gena               3
pueen              1
Name: count, Length: 273, dtype: int64

In [15]:
df_cleaned.event_type.value_counts()


event_type
view                9656759
cart                5649646
remove_from_cart    2987150
purchase            1286102
Name: count, dtype: int64

### Changing types and checking for event_time correctness

In [16]:
df_cleaned['event_type'] = df_cleaned['event_type'].astype('category')
df_cleaned['brand'] = df_cleaned['brand'].fillna('unknown')
df_cleaned['brand'] = df_cleaned['brand'].astype('category')


In [17]:
df_cleaned['event_time_raw'] = df_cleaned['event_time']
df_cleaned['event_time'] = pd.to_datetime(
                        df_cleaned['event_time'], 
                        utc=True, 
                        errors='coerce')

Using `errors='coerce'` to safely convert datetime and validate that no values were lost.

In [18]:
df_cleaned['event_time'].describe()

count                               19579657
mean     2019-12-16 04:13:25.977567744+00:00
min                2019-10-01 00:00:00+00:00
25%                2019-11-08 12:54:21+00:00
50%                2019-12-12 21:47:55+00:00
75%                2020-01-25 10:25:46+00:00
max                2020-02-29 23:59:59+00:00
Name: event_time, dtype: object

In [19]:
df_cleaned.event_time.isna().sum()

np.int64(0)

As wronly converted values not found and values for dates are correct, we can conclude that event_time converted correctly. Thus, dropping event_time_raw.

In [20]:
df_cleaned = df_cleaned.drop(columns=['event_time_raw'])

### Adding event_time details (hours, weekday, month, year, is_weekend, day_number, and week_number)

In [21]:
df_time_splits = df_cleaned.copy()
df_time_splits['hour'] = df_cleaned.event_time.dt.hour
df_time_splits['weekday'] = df_time_splits.event_time.dt.dayofweek
df_time_splits['is_weekend'] = df_time_splits['weekday'].isin([5,6])
df_time_splits['month'] = df_time_splits.event_time.dt.month
df_time_splits['year'] = df_time_splits.event_time.dt.year


In [22]:
min_date = df_time_splits.event_time.min().date()
df_time_splits['day_number'] = (df_time_splits.event_time.dt.date - min_date).apply(lambda x: x.days + 1)
df_time_splits['week_number'] = (df_time_splits.day_number//7)+1 

In [23]:
df_time_splits.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19579657 entries, 0 to 20692839
Data columns (total 15 columns):
 #   Column        Dtype              
---  ------        -----              
 0   event_time    datetime64[ns, UTC]
 1   event_type    category           
 2   product_id    int32              
 3   category_id   int64              
 4   brand         category           
 5   price         float64            
 6   user_id       int32              
 7   user_session  object             
 8   hour          int32              
 9   weekday       int32              
 10  is_weekend    bool               
 11  month         int32              
 12  year          int32              
 13  day_number    int64              
 14  week_number   int64              
dtypes: bool(1), category(2), datetime64[ns, UTC](1), float64(1), int32(6), int64(3), object(1)
memory usage: 1.5+ GB


In [24]:
df_time_splits.head(10)

,event_time,event_type,product_id,category_id,brand,price,user_id,user_session,hour,weekday,is_weekend,month,year,day_number,week_number
0,2019-12-01 00:00:00+00:00,remove_from_cart,5712790,1487580005268456287,f.o.x,6.27,576802932,51d85cb0-897f-48d2-918b-ad63965c12dc,0,6,True,12,2019,62,9
1,2019-12-01 00:00:00+00:00,view,5764655,1487580005411062629,cnd,29.05,412120092,8adff31e-2051-4894-9758-224bfa8aec18,0,6,True,12,2019,62,9
2,2019-12-01 00:00:02+00:00,cart,4958,1487580009471148064,runail,1.19,494077766,c99a50e8-2fac-4c4d-89ec-41c05f114554,0,6,True,12,2019,62,9
3,2019-12-01 00:00:05+00:00,view,5848413,1487580007675986893,freedecor,0.79,348405118,722ffea5-73c0-4924-8e8f-371ff8031af4,0,6,True,12,2019,62,9
4,2019-12-01 00:00:07+00:00,view,5824148,1487580005511725929,unknown,5.56,576005683,28172809-7e4a-45ce-bab0-5efa90117cd5,0,6,True,12,2019,62,9
5,2019-12-01 00:00:09+00:00,view,5773361,1487580005134238553,runail,2.62,560109803,38cf4ba1-4a0a-4c9e-b870-46685d105f95,0,6,True,12,2019,62,9
6,2019-12-01 00:00:18+00:00,cart,5629988,1487580009311764506,unknown,1.19,579966747,1512be50-d0fd-4a92-bcd8-3ea3943f2a3b,0,6,True,12,2019,62,9
7,2019-12-01 00:00:22+00:00,view,5807805,1487580005713052531,ingarden,4.44,576005683,28172809-7e4a-45ce-bab0-5efa90117cd5,0,6,True,12,2019,62,9
8,2019-12-01 00:00:27+00:00,view,5588608,1487580008145748965,roubloff,5.40,546170008,676d9fcc-2a4f-4448-b49d-136f2e4208c1,0,6,True,12,2019,62,9
9,2019-12-01 00:00:34+00:00,cart,5335,1487580009605365797,runail,0.40,494077766,c99a50e8-2fac-4c4d-89ec-41c05f114554,0,6,True,12,2019,62,9


Everything looks

### Dealing with negative prices

Removing by such condition:
- Negative prices should be removed
- Purchase price cannot be zero

In [25]:
negative_price_mask = df_time_splits.price < 0
zero_price_mask = (df_time_splits.price == 0) & (df_time_splits.event_type == 'purchase')
price_mask = negative_price_mask | zero_price_mask
df_corrected_prices = df_time_splits.loc[~price_mask, :]

In [26]:
df_time_splits.loc[price_mask, :].shape

(125, 15)

As we can see, 125 rows is unissential for the dataset, so it is removed from df.

### Exporting dataset to csv for further investigation

In [27]:
df_corrected_prices.to_csv('cleaned_data.csv', index=False)